In [1]:
import pandas as pd
import numpy as np

In [2]:
business_interruption_file_path = "dataset/srcsc-2026-claims-business-interruption.xlsx"
claim_cargo_file_path = "dataset/srcsc-2026-claims-cargo.xlsx"
claims_equipment_failure_file_path = "dataset/srcsc-2026-claims-equipment-failure.xlsx"
claims_workers_comp_file_path = "dataset/srcsc-2026-claims-workers-comp.xlsx"

In [3]:
solar_system_names = [
    'Helionis Cluster',
    'Epsilon',
    'Zeta'
]

In [ ]:
def ratio(df, column_name):
    """
    Cleans a column to bring values toward a 0-1 range:
    - 0 to 1: No change
    - 0 to -1: Absolute value
    - > 1: Divide by 100
    - < -1: Absolute value, then Divide by 100
    """
    # Use a local variable for readability
    col = df[column_name]
    
    conditions = [
        (col >= 0) & (col <= 1),   # Pass
        (col < 0) & (col >= -1),  # Small negative -> Absolute
        (col < -1),               # Large negative -> /100
        (col > 1)                 # Large positive -> /100
    ]

    choices = [
        col,                # Keep original
        col.abs(),          # Absolute value
        (col / 100).abs(),    # Divide by 100
        col / 100           # Divide by 100
    ]

    # Apply logic and return the modified column (or update df)
    df[column_name] = np.select(conditions, choices, default=col)
    
    return df

def clean_int_range(df, column_name, start, end):
    # 1. Absolute value to fix negatives
    df[column_name] = df[column_name].abs()
    
    # 2. Round to the nearest integer (e.g., 3.7 -> 4.0)
    # Use .round(0) then convert to Int64 to handle potential NaNs safely
    df[column_name] = df[column_name].round(0)
    
    # 3. Constrain to the 0-4 range
    df[column_name] = df[column_name].clip(lower = start, upper = end)
    
    # 4. Convert to integer type
    # 'Int64' (capital I) allows for NaN values, unlike 'int64'
    df[column_name] = df[column_name].astype('Int64')
    
    return df

def clean_business_interruption_data(df, name = solar_system_names):
    """
    Unified cleaning function that handles both Sheet 1 and Sheet 2 
    by checking for column existence.
    """
    
    # 1. ID Cleaning (String Truncation)
    if 'claim_id' in df.columns:
        df['claim_id'] = df['claim_id'].str[:12]
        
    if 'policy_id' in df.columns:
        df['policy_id'] = df['policy_id'].str[:9]
        
    if 'station_id' in df.columns:
        df['station_id'] = df['station_id'].str[:2]

    # 2. Categorical Cleaning (Regex)
    if 'solar_system' in df.columns:
        df['solar_system'] = df['solar_system'].str.extract(
            f"({'|'.join(solar_system_names)})",
            expand=False
        )
    
    # 3. Financial Cleaning
    if 'claim_amount' in df.columns:
        # Ensures all negative claims (likely entry errors) are positive
        df['claim_amount'] = df['claim_amount'].abs()

    # 4. Ratio Columns (0 to 1 Logic)
    ratio_columns = ['production_load', 'supply_chain_index', 'exposure']
    for col in ratio_columns:
        if col in df.columns:
            df = ratio(df, col)

    # 5. Integer Range Columns (Custom Min/Max)
    # We use a list of tuples to store (column_name, start, end)
    range_rules = [
        ('energy_backup_score', 1, 5),
        ('avg_crew_exp', 1, 30),
        ('maintenance_freq', 0, 6),
        ('safety_compliance', 1, 5),
        ('claim_count', 0, 4)
    ]
    
    for col, start, end in range_rules:
        if col in df.columns:
            # Note: renamed 'range' to 'clean_int_range' to avoid conflict
            df = clean_int_range(df, col, start, end)

    return df

In [5]:
df = pd.read_excel(business_interruption_file_path)
df = clean_business_interruption_data(df)

In [ ]:
df

In [6]:
df_2 = pd.read_excel(business_interruption_file_path, sheet_name = 1)
df_2 = clean_business_interruption_data(df_2)

In [ ]:
df_2